# Cabra wearable: servos (PCA9685) + IMUs

Notebook para el firmware `ErisServoIMU` (flavor ErisServoDriver de Eris + 3 MPU6050).

1. **Conexión**: abre el puerto, espera la calibración de los IMUs y activa el streaming `S_F IMU`.
2. **Trayectoria**: carga un CSV y lo convierte a ángulos de servo con `MAPEO`.
3. **Envío**: los puntos van a una cola en la ESP32, que los reproduce a periodo fijo (200 Hz).
4. **Gráfica en vivo**: ángulos de los IMUs y lo que se está mandando a los servos.
   Teclas en la ventana: `c` = nuevo cero de IMUs, `t` = reproducir, `x` = detener, `i` = ir al inicio.
5. **Análisis**: IMUs medidos vs. trayectoria comandada, desde el CSV guardado.

Comandos del firmware: `X ch ang`, `Y ch ang` (suave), `T_CH`, `T_DT`, `P`, `T_GO`, `T_END`, `T_STOP`, `T_ST`, `CAL`, `IMU`.

In [ ]:
%matplotlib qt
import re
import csv
import time
import struct
import threading
from collections import deque
import numpy as np
import pandas as pd
import serial
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# ---------------- Configuración ----------------
PUERTO = "COM5"
BAUDIOS = 921600                  # ERIS_SERIAL_BAUD del firmware en ESP32
ARTICULACIONES = ["hip", "knee", "ankle"]
ARCHIVO_CSV = f"imus_servos_{time.strftime('%Y%m%d_%H%M%S')}.csv"

# JointSample_t del firmware: t_ms, hip, knee, ankle, traj_idx, traj_cola
MUESTRA = struct.Struct("<6f")


def cobs_decode(datos):
    """Decodifica una trama COBS (PacketSerial) sin el 0x00 final."""
    salida = bytearray()
    i, n = 0, len(datos)
    while i < n:
        codigo = datos[i]
        if codigo == 0 or i + codigo > n:
            raise ValueError("trama COBS inválida")
        salida += datos[i + 1:i + codigo]
        i += codigo
        if codigo < 0xFF and i < n:
            salida.append(0)
    return bytes(salida)


# Líneas de texto (Serial.println del firmware) que llegan mezcladas con las tramas binarias
_TEXTO = re.compile(rb"(?:[\t\x20-\x7e]+\r?\n)+")


class Eris:
    """Conexión con el firmware ErisServoIMU.

    Un hilo lee el puerto sin parar: separa las tramas COBS (terminadas en 0x00)
    del texto plano, guarda las muestras de los IMUs en `muestras` y en el CSV,
    y deja los mensajes de texto en `mensajes`.
    """

    def __init__(self, puerto, baudios, archivo_csv, eco=True):
        self.ser = serial.Serial(puerto, baudios, timeout=0.05)
        self.eco = eco
        self.lock = threading.Lock()          # protege muestras / estado
        self.lock_tx = threading.Lock()       # un solo escritor a la vez en el puerto
        self.muestras = deque(maxlen=500_000) # (t_s, hip, knee, ankle, idx, cola)
        self.mensajes = deque(maxlen=500)     # (n, texto)
        self.n_mensajes = 0
        self.ultimo_idx = -1                  # puntos reproducidos (-1 = sin trayectoria)
        self.ultima_cola = 0
        self.tramas_malas = 0
        self.tray_enviada = None              # array (N, k) que se está reproduciendo
        self._buf = b""
        self._corriendo = True

        self.archivo = open(archivo_csv, "w", newline="")
        self.escritor = csv.writer(self.archivo)
        self.escritor.writerow(["t_s"] + ARTICULACIONES + ["traj_idx", "traj_cola"])

        self.hilo = threading.Thread(target=self._leer, daemon=True)
        self.hilo.start()

    # ---------- envío ----------
    def enviar(self, comando):
        self.enviar_lineas([comando])

    def enviar_lineas(self, lineas):
        with self.lock_tx:
            self.ser.write(("\n".join(lineas) + "\n").encode())

    def marca(self):
        """Número del próximo mensaje; tómala ANTES de enviar el comando que esperas."""
        return self.n_mensajes

    def esperar_texto(self, fragmento, desde, timeout=5.0):
        """Espera un mensaje (número >= desde) que contenga `fragmento`. True si llegó."""
        t_fin = time.time() + timeout
        while time.time() < t_fin:
            with self.lock:
                if any(n >= desde and fragmento in m for n, m in self.mensajes):
                    return True
            time.sleep(0.02)
        return False

    # ---------- recepción ----------
    def _texto(self, crudo):
        for linea in crudo.decode(errors="replace").splitlines():
            linea = linea.strip()
            if linea:
                with self.lock:
                    self.mensajes.append((self.n_mensajes, linea))
                    self.n_mensajes += 1
                if self.eco:
                    print("<-", linea)

    def _paquete(self, trama):
        """Intenta interpretar la trama como paquete Eris. True si era válido."""
        try:
            d = cobs_decode(trama)
        except ValueError:
            return False
        tipo = d[:1]
        if tipo == b"D" and len(d) >= 2:
            n = d[1]
            if len(d) != 2 + n * MUESTRA.size:
                return False
            filas = [MUESTRA.unpack_from(d, 2 + k * MUESTRA.size) for k in range(n)]
            with self.lock:
                for t_ms, h, c, ca, idx, cola in filas:
                    self.muestras.append((t_ms / 1000, h, c, ca, idx, cola))
                    self.escritor.writerow([f"{t_ms / 1000:.4f}", f"{h:.2f}", f"{c:.2f}",
                                            f"{ca:.2f}", int(idx), int(cola)])
                if filas:
                    self.ultimo_idx = int(filas[-1][4])
                    self.ultima_cola = int(filas[-1][5])
            return True
        if tipo in (b"T", b"E"):
            self._texto((b"ERROR: " if tipo == b"E" else b"") + d[1:].replace(b"\0", b" ") + b"\n")
            return True
        return False

    def _trama(self, trama):
        if not trama or self._paquete(trama):
            return
        # Texto pegado delante de un paquete (el firmware imprime texto plano con Serial.println)
        m = _TEXTO.match(trama)
        if m:
            self._texto(m.group())
            resto = trama[m.end():]
            if not resto or self._paquete(resto):
                return
        self.tramas_malas += 1

    def _procesar(self, datos):
        self._buf += datos
        *tramas, self._buf = self._buf.split(b"\0")
        for trama in tramas:
            self._trama(trama)
        # Texto completo sin paquete detrás (p. ej. respuestas con el streaming apagado)
        if self._buf.endswith(b"\n") and _TEXTO.fullmatch(self._buf):
            self._texto(self._buf)
            self._buf = b""

    def _leer(self):
        t_flush = time.time()
        while self._corriendo:
            try:
                datos = self.ser.read(max(1, self.ser.in_waiting))
            except serial.SerialException:
                break
            if datos:
                self._procesar(datos)
            if time.time() - t_flush > 1:
                self.flush()
                t_flush = time.time()

    def flush(self):
        with self.lock:
            self.archivo.flush()

    def cerrar(self):
        self._corriendo = False
        self.hilo.join(timeout=1)
        try:
            self.enviar("T_STOP")
            self.enviar("S_OFF")
        except serial.SerialException:
            pass
        self.ser.close()
        self.archivo.close()


# ---------------- Conexión ----------------
try:
    eris.cerrar()        # si re-ejecutas la celda, cierra la conexión anterior
except NameError:
    pass

eris = Eris(PUERTO, BAUDIOS, ARCHIVO_CSV)
print(f"Conectado a {PUERTO}. La ESP32 se reinicia y calibra: deja la pata QUIETA...")
if not eris.esperar_texto("Serial Commands are ready", desde=0, timeout=12):
    print("(no llegó el mensaje de arranque; continúo de todos modos)")

eris.enviar("S_F IMU")
eris.enviar("S_TIME")
eris.enviar("S_ON")
time.sleep(0.5)
print(f"Streaming activo, {len(eris.muestras)} muestras recibidas. Guardando en {ARCHIVO_CSV}")

## Trayectoria

Solo se usan las patas delanteras: `fr` (derecha) o `fl` (izquierda), que es la misma curva
desfasada medio ciclo. Hip, knee y ankle van a los canales 0, 1 y 2 de la PCA9685.

`servo = offset + signo * ángulo_del_CSV`, recortado a 0–180°. Ajusta `signo` y `offset` a cómo
montaste cada servo.

In [ ]:
ARCHIVO_TRAY = "leg_actuator_angles_one_cycle.csv"   # o "leg_actuator_angles_trajectory_adapted.csv"
REPETICIONES = 10                                    # veces que se repite el archivo

PIERNA = "fr"                                        # "fr" o "fl"

# articulación: (canal PCA9685, signo, offset °)
MAPEO = {
    "hip":   (0, +1, 0.0),
    "knee":  (1, +1, 0.0),
    "ankle": (2, +1, 0.0),
}

tray = pd.read_csv(ARCHIVO_TRAY)
DT_S = float(np.median(np.diff(tray["time_s"])))
columnas = [f"{PIERNA}_{a}_joint" for a in MAPEO]
canales = [MAPEO[a][0] for a in MAPEO]
signo = np.array([MAPEO[a][1] for a in MAPEO], dtype=float)
offset = np.array([MAPEO[a][2] for a in MAPEO], dtype=float)

servo = offset + signo * tray[columnas].to_numpy()
for j, c in enumerate(columnas):
    fuera = np.mean((servo[:, j] < 0) | (servo[:, j] > 180))
    if fuera > 0:
        print(f"AVISO: {c} (canal {canales[j]}) sale de 0-180° en el {fuera:.0%} de los puntos; se recorta")
servo = np.clip(servo, 0, 180)
servo_env = np.tile(servo, (REPETICIONES, 1))

print(f"{ARCHIVO_TRAY}: {len(tray)} puntos, dt = {DT_S * 1000:.1f} ms, "
      f"{len(servo_env)} puntos a enviar ({len(servo_env) * DT_S:.1f} s)")

fig, axs = plt.subplots(2, 1, sharex=True, figsize=(10, 6))
t = tray["time_s"].to_numpy()
for j, c in enumerate(columnas):
    axs[0].plot(t, tray[c], label=c)
    axs[1].plot(t, servo[:, j], label=f"{list(MAPEO)[j]} (ch {canales[j]})")
axs[0].set_ylabel("Ángulo articular (°)")
axs[1].set_ylabel("Ángulo de servo (°)")
axs[1].set_xlabel("Tiempo (s)")
axs[0].legend(fontsize=8)
axs[1].legend(fontsize=8)
for ax in axs:
    ax.grid(True, alpha=0.3)
fig.suptitle(f"Una repetición de la trayectoria ({PIERNA})")
plt.show()

## Envío de la trayectoria

El envío corre en un hilo aparte para que la gráfica en vivo siga funcionando.
Se mantienen `ADELANTO` puntos en la cola de la ESP32 (la cola usa lo que reporta el propio streaming),
así que el ritmo lo marca el reloj de la ESP32 y no el de Windows.

In [ ]:
ADELANTO = 200          # puntos en cola en la ESP32 (1 s a 200 Hz; la cola admite 511)
MAX_LINEA = 64          # SERIALCOMMAND_BUFFER del firmware

_envio = {"hilo": None, "parar": False}


def linea_punto(fila):
    return "P " + " ".join(str(int(round(v * 10))) for v in fila)   # décimas de grado


def ir_a_inicio(esperar=2.0):
    """Lleva cada servo suavemente (comando Y) al primer punto de la trayectoria."""
    eris.enviar_lineas([f"Y {ch} {v:.1f}" for ch, v in zip(canales, servo_env[0])])
    time.sleep(esperar)


def _enviar_trayectoria():
    lineas = [linea_punto(f) for f in servo_env]
    largo = max(map(len, lineas))
    if largo > MAX_LINEA:
        print(f"ERROR: las líneas P miden {largo} caracteres (máx. {MAX_LINEA}); usa menos columnas")
        return

    marca = eris.marca()
    eris.enviar_lineas(["T_STOP", f"T_DT {DT_S * 1000:g}", "T_CH " + " ".join(map(str, canales))])
    if not eris.esperar_texto("T_CH ok", marca, timeout=2):
        print("ERROR: el firmware no confirmó T_CH")
        return
    with eris.lock:
        eris.ultimo_idx = -1
    eris.tray_enviada = servo_env

    n_total = len(lineas)
    enviados = min(ADELANTO, n_total)
    eris.enviar_lineas(lineas[:enviados])      # precarga
    eris.enviar("T_GO")
    t_inicio = time.time()

    while enviados < n_total and not _envio["parar"]:
        reproducidos = max(eris.ultimo_idx, 0)
        falta = ADELANTO - (enviados - reproducidos)
        if falta > 0:
            bloque = lineas[enviados:enviados + falta]
            eris.enviar_lineas(bloque)
            enviados += len(bloque)
        time.sleep(0.01)

    if _envio["parar"]:
        eris.enviar("T_STOP")
        print(f"Trayectoria detenida en el punto {eris.ultimo_idx} de {n_total}")
        return

    marca = eris.marca()
    eris.enviar("T_END")
    restante = (n_total - max(eris.ultimo_idx, 0)) * DT_S
    if not eris.esperar_texto("Trayectoria terminada", marca, timeout=restante + 5):
        print("AVISO: no llegó 'Trayectoria terminada'")
    print(f"Trayectoria completa: {n_total} puntos en {time.time() - t_inicio:.2f} s "
          f"(esperado {n_total * DT_S:.2f} s)")
    eris.enviar("T_ST")     # underruns / rechazados


def reproducir(desde_inicio=True):
    """Arranca la trayectoria en segundo plano."""
    if _envio["hilo"] is not None and _envio["hilo"].is_alive():
        print("Ya hay una trayectoria en curso (usa detener())")
        return

    def tarea():
        if desde_inicio:
            ir_a_inicio()
        _enviar_trayectoria()

    _envio["parar"] = False
    _envio["hilo"] = threading.Thread(target=tarea, daemon=True)
    _envio["hilo"].start()


def detener():
    _envio["parar"] = True
    eris.enviar("T_STOP")

## Gráfica en vivo

Arriba: ángulos de los IMUs. Abajo: ángulo que la ESP32 está mandando a cada servo.
Teclas con la ventana activa: `c` cero de IMUs, `t` reproducir, `x` detener, `i` ir al inicio.

In [ ]:
VENTANA_S = 10

fig, (ax_imu, ax_srv) = plt.subplots(2, 1, sharex=True, figsize=(11, 7))
lineas_imu = {a: ax_imu.plot([], [], label=a)[0] for a in ARTICULACIONES}
lineas_srv = [ax_srv.plot([], [], label=f"{a} (ch {ch})")[0] for a, ch in zip(MAPEO, canales)]
ax_imu.set_ylabel("Ángulo IMU (°)")
ax_srv.set_ylabel("Comando servo (°)")
ax_srv.set_xlabel("Tiempo ESP32 (s)")
ax_srv.set_ylim(-5, 185)
ax_imu.legend(loc="upper left")
ax_srv.legend(loc="upper left")
for ax in (ax_imu, ax_srv):
    ax.grid(True, alpha=0.3)
titulo = fig.suptitle("")


def ventana():
    """Muestras de los últimos VENTANA_S segundos (más antiguas primero)."""
    with eris.lock:
        if not eris.muestras:
            return None
        t_fin = eris.muestras[-1][0]
        recientes = []
        for m in reversed(eris.muestras):
            if m[0] < t_fin - VENTANA_S:
                break
            recientes.append(m)
    return np.array(recientes[::-1])


def actualizar(_):
    datos = ventana()
    if datos is None or len(datos) < 2:
        return []
    t = datos[:, 0]
    for k, a in enumerate(ARTICULACIONES):
        lineas_imu[a].set_data(t, datos[:, 1 + k])

    # punto aplicado = idx - 1 dentro de la trayectoria enviada
    idx = datos[:, 4].astype(int) - 1
    tray_env = eris.tray_enviada
    for j, linea in enumerate(lineas_srv):
        if tray_env is not None and j < tray_env.shape[1]:
            valido = (idx >= 0) & (idx < len(tray_env))
            y = np.full(len(t), np.nan)
            y[valido] = tray_env[idx[valido], j]
            linea.set_data(t, y)

    ax_imu.set_xlim(max(0, t[-1] - VENTANA_S), max(VENTANA_S, t[-1]))
    ang = datos[:, 1:4]
    ax_imu.set_ylim(ang.min() - 5, ang.max() + 5)
    estado = f"trayectoria: punto {eris.ultimo_idx}" if eris.ultimo_idx >= 0 else "trayectoria: parada"
    titulo.set_text(f"{estado} | cola ESP32: {eris.ultima_cola} | tramas malas: {eris.tramas_malas}")
    return list(lineas_imu.values()) + lineas_srv


def al_teclear(evento):
    if evento.key == "c":
        eris.enviar("CAL")
    elif evento.key == "t":
        reproducir()
    elif evento.key == "x":
        detener()
    elif evento.key == "i":
        threading.Thread(target=ir_a_inicio, daemon=True).start()


for tecla in "ctxi":        # quita los atajos de matplotlib que usan estas teclas
    for nombre in ("keymap.back", "keymap.forward", "keymap.home", "keymap.pan", "keymap.zoom"):
        if tecla in plt.rcParams[nombre]:
            plt.rcParams[nombre].remove(tecla)
fig.canvas.mpl_connect("key_press_event", al_teclear)
anim = FuncAnimation(fig, actualizar, interval=40, cache_frame_data=False)
plt.show()

## Control manual

También se puede lanzar todo desde aquí sin la gráfica.

In [ ]:
# reproducir()                  # ir al inicio + trayectoria completa en segundo plano
# detener()
# eris.enviar("CAL")            # nueva pose neutra de los IMUs
# eris.enviar("T_ST")           # estado de la cola: underruns, rechazados...
# eris.enviar("Y 0 90")         # servo 0 a 90° suave
# eris.enviar("X 0 90")         # servo 0 a 90° inmediato
eris.enviar("T_ST")

## Análisis: IMUs vs. trayectoria comandada

Relaciona cada articulación medida con una columna del CSV. Como los IMUs miden respecto
a la pose neutra y la trayectoria es absoluta, se comparan quitando la media de cada señal.

In [ ]:
COMPARAR = {a: f"{PIERNA}_{a}_joint" for a in ARTICULACIONES}   # IMU de cada eslabón vs. su columna

eris.flush()
reg = pd.read_csv(ARCHIVO_CSV)
activo = reg[reg["traj_idx"] >= 1].copy()
if activo.empty:
    print("Todavía no hay datos grabados durante una trayectoria")
else:
    articulares = np.tile(tray[list(COMPARAR.values())].to_numpy(), (REPETICIONES, 1))
    punto = (activo["traj_idx"].to_numpy() - 1).clip(0, len(articulares) - 1)

    fig, axs = plt.subplots(len(COMPARAR), 1, sharex=True, figsize=(11, 8))
    for ax, (imu, col), j in zip(axs, COMPARAR.items(), range(len(COMPARAR))):
        medido = activo[imu].to_numpy()
        comando = articulares[punto, j]
        ax.plot(activo["t_s"], medido - medido.mean(), label=f"IMU {imu}")
        ax.plot(activo["t_s"], comando - comando.mean(), "--", label=f"comando {col}")
        ax.set_ylabel("Δ ángulo (°)")
        ax.legend(loc="upper right")
        ax.grid(True, alpha=0.3)
    axs[-1].set_xlabel("Tiempo ESP32 (s)")
    fig.suptitle(ARCHIVO_CSV)
    plt.show()

In [ ]:
eris.cerrar()
print("Puerto cerrado")